# 🔬 Datenanalyse-Agent mit LangGraph – Stufe 2
---

## Überblick

Dieses Notebook implementiert einen **agentischen Datenanalyse-Assistenten** 
mit **Reflexionsschleife** und **austauschbaren Skills**. 

### Architektur

```
START → data_profiler → [skill_auswahl] → analyst ⇄ tools → quality_checker ──→ END
                                                                    └──→ analyst (max. 3×)
```

| Knoten | Typ | Aufgabe |
|--------|-----|---------|
| **Data Profiler** | Deterministisch | Datei einlesen, Struktur erfassen, **Skills auswählen** |
| **Analyst** | LLM + Tool-Calling | Analysen durchführen (gesteuert durch geladene Skills) |
| **Tools** | Code-Sandbox | Python-Code sicher ausführen |
| **Quality Checker** | LLM (günstigeres Modell) | Bewertung **gegen die aktiven Skills** |

### Neue Konzepte gegenüber dem Basis-Notebook
- **Skills als .md-Dateien** — Prompt-Engineering als Konfiguration statt hardcoded String
- **Automatische Skill-Auswahl** — Data Profiler wählt passende Skills basierend auf Spaltentypen
- **Reflexionsschleife** — Quality Checker bewertet und gibt strukturiertes Feedback
- **Zwei LLMs** — Analyst (leistungsfähig) ≠ Checker (günstig)

### Skills-Ordner

```
skills/
├── analyst/
│   ├── _basis.md              ← immer geladen (Grundregeln, Code-Konventionen)
│   ├── numerisch.md           ← bei numerischen Spalten
│   ├── kategorisch.md         ← bei kategorischen Spalten
│   ├── zeitreihe.md           ← bei Datumsspalten
│   └── fehlende_werte.md      ← bei fehlenden Werten
├── checker/
│   └── standard.md            ← Bewertungskriterien
└── README.md                  ← Anleitung für eigene Skills
```

**Kernidee:** Die Skills definieren, *was* analysiert werden soll. 
Der LLM entscheidet, *wie*. Studenten können Skills anpassen oder 
neue erstellen, ohne Python-Code zu ändern.

## 1. Installation & Setup

In [ ]:
# Abhängigkeiten installieren (einmalig ausführen)
# !pip install langgraph langchain-openai pandas openpyxl matplotlib seaborn gradio

In [ ]:
import os
import sys
import json
import re
import io
import contextlib
from pathlib import Path
from typing import TypedDict, Annotated

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import getpass

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("✓ Alle Bibliotheken geladen")

## 2. Konfiguration & Modelle

Alle konfigurierbaren Parameter an einer Stelle. Der `SKILLS_DIR`-Pfad 
zeigt auf den `skills/`-Ordner neben dem Notebook.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Konfiguration
# ══════════════════════════════════════════════════════════════

# Modelle (DeepInfra-kompatible Modellnamen)
ANALYST_MODEL = "Qwen/Qwen3.6-35B-A3B"
CHECKER_MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning"

# Qualitäts-Threshold und Retry-Limit
QUALITY_THRESHOLD = 0.70
MAX_REVISIONS = 3

# Score-Gewichtung
WEIGHT_COMPLETENESS = 0.60
WEIGHT_DEPTH = 0.40

# Skills-Verzeichnis (relativ zum Notebook)
SKILLS_DIR = Path("skills")

# ══════════════════════════════════════════════════════════════
# API-Key & Modelle
# ══════════════════════════════════════════════════════════════
DEEPINFRA_API_KEY = getpass.getpass("DeepInfra API-Key eingeben: ")

llm_analyst = ChatOpenAI(
    model_name=ANALYST_MODEL,
    openai_api_key=DEEPINFRA_API_KEY,
    openai_api_base="https://api.deepinfra.com/v1/openai",
    max_tokens=5000,
    temperature=0,
)

llm_checker = ChatOpenAI(
    model_name=CHECKER_MODEL,
    openai_api_key=DEEPINFRA_API_KEY,
    openai_api_base="https://api.deepinfra.com/v1/openai",
    max_tokens=2000,
    temperature=0,
)

print(f"✓ Analyst-Modell:  {ANALYST_MODEL}")
print(f"✓ Checker-Modell:  {CHECKER_MODEL}")
print(f"✓ Skills-Ordner:   {SKILLS_DIR.resolve()}")
print(f"✓ Threshold:       {QUALITY_THRESHOLD:.0%}")

## 3. Skill-Loader

Die zentrale Funktion zum Laden und Befüllen von Skills. Jede `.md`-Datei 
kann Platzhalter wie `{file_path}` oder `{numerische_spalten}` enthalten, 
die zur Laufzeit mit echten Werten aus dem Datenprofil ersetzt werden.

Die Funktion `select_analyst_skills` entscheidet basierend auf dem 
Datenprofil, welche Skills geladen werden. Die Logik ist bewusst einfach 
gehalten — man kann sie erweitern, z.B. um domänenspezifische Skills 
anhand von Spaltennamen zu erkennen.

In [ ]:
def load_skill(skill_path: Path, context: dict) -> str:
    """
    Lädt eine Skill-Datei (.md) und ersetzt Platzhalter.
    
    Args:
        skill_path: Pfad zur .md-Datei
        context: Dict mit Platzhalter-Werten (z.B. file_path, spalten, ...)
        
    Returns:
        Aufbereiteter Skill-Text, oder leerer String bei Fehler
    """
    if not skill_path.exists():
        print(f"  ⚠️ Skill nicht gefunden: {skill_path}")
        return ""
    
    text = skill_path.read_text(encoding="utf-8")
    
    # Platzhalter ersetzen (fehlende Platzhalter bleiben stehen)
    for key, value in context.items():
        placeholder = "{" + key + "}"
        if placeholder in text:
            text = text.replace(placeholder, str(value))
    
    return text


def select_analyst_skills(profile: dict) -> list[str]:
    """
    Wählt Analyst-Skills basierend auf dem Datenprofil aus.
    
    Returns:
        Liste der Skill-Dateinamen (ohne Pfad), z.B. ['numerisch.md', 'zeitreihe.md']
    """
    skills = []
    
    # Numerische Spalten vorhanden?
    if profile.get("numerische_spalten"):
        skills.append("numerisch.md")
    
    # Kategorische Spalten vorhanden?
    if profile.get("kategorische_spalten"):
        skills.append("kategorisch.md")
    
    # Datumsspalten erkennen (heuristisch: Spaltenname enthält 'datum', 'date', 'zeit', 'time')
    datum_keywords = {"datum", "date", "zeit", "time", "tag", "monat", "jahr", "year", "month", "day"}
    all_cols = [col.lower() for col in profile.get("spaltennamen", [])]
    if any(keyword in col for col in all_cols for keyword in datum_keywords):
        skills.append("zeitreihe.md")
    
    # Fehlende Werte vorhanden?
    if profile.get("fehlende_werte"):
        skills.append("fehlende_werte.md")
    
    return skills


def build_analyst_prompt(profile: dict, file_path: str, active_skills: list[str]) -> str:
    """
    Baut den Analyst-System-Prompt aus Skills zusammen.
    
    1. Lädt immer _basis.md
    2. Lädt alle aktiven Spezial-Skills
    3. Fügt Platzhalter-Werte ein
    """
    # Kontext für Platzhalter
    context = {
        "file_path": file_path,
        "profile_json": json.dumps(profile, indent=2, ensure_ascii=False),
        "spalten": ", ".join(profile.get("spaltennamen", [])),
        "numerische_spalten": ", ".join(profile.get("numerische_spalten", [])),
        "kategorische_spalten": ", ".join(profile.get("kategorische_spalten", [])),
        "fehlende_werte": json.dumps(profile.get("fehlende_werte", {}), ensure_ascii=False),
        "zeilen": str(profile.get("zeilen", "?")),
    }
    
    # Basis-Skill (immer)
    parts = []
    basis = load_skill(SKILLS_DIR / "analyst" / "_basis.md", context)
    if basis:
        parts.append(basis)
    else:
        # Fallback falls _basis.md fehlt
        parts.append(f"Du bist ein Datenanalyst. Datenprofil:\n{context['profile_json']}")
    
    # Spezial-Skills
    for skill_name in active_skills:
        skill_text = load_skill(SKILLS_DIR / "analyst" / skill_name, context)
        if skill_text:
            parts.append(skill_text)
    
    return "\n\n---\n\n".join(parts)


def build_checker_prompt(profile: dict, active_skills: list[str], conversation: str) -> str:
    """
    Baut den Checker-Prompt aus dem Checker-Skill zusammen.
    """
    context = {
        "spalten": ", ".join(profile.get("spaltennamen", [])),
        "numerische_spalten": ", ".join(profile.get("numerische_spalten", [])),
        "kategorische_spalten": ", ".join(profile.get("kategorische_spalten", [])),
        "fehlende_werte": json.dumps(profile.get("fehlende_werte", {}), ensure_ascii=False),
        "zeilen": str(profile.get("zeilen", "?")),
        "aktive_skills": ", ".join(active_skills) if active_skills else "Keine speziellen Skills",
    }
    
    checker_text = load_skill(SKILLS_DIR / "checker" / "standard.md", context)
    
    if not checker_text:
        # Fallback falls Checker-Skill fehlt
        checker_text = (
            f"Bewerte die Analyse. Spalten: {context['spalten']}. "
            f"Aktive Skills: {context['aktive_skills']}."
        )
    
    return checker_text + f"\n\n## Analyseverlauf\n\n{conversation[-4000:]}"


# ── Skills-Ordner prüfen ──
if SKILLS_DIR.exists():
    analyst_skills = list((SKILLS_DIR / "analyst").glob("*.md")) if (SKILLS_DIR / "analyst").exists() else []
    checker_skills = list((SKILLS_DIR / "checker").glob("*.md")) if (SKILLS_DIR / "checker").exists() else []
    print(f"✓ Skills gefunden: {len(analyst_skills)} Analyst, {len(checker_skills)} Checker")
    for s in analyst_skills:
        print(f"  📄 analyst/{s.name}")
    for s in checker_skills:
        print(f"  📄 checker/{s.name}")
else:
    print(f"⚠️ Skills-Ordner '{SKILLS_DIR}' nicht gefunden!")
    print(f"  Der Agent funktioniert trotzdem (mit Fallback-Prompts),")
    print(f"  aber Skills werden nicht geladen.")

## 4. State definieren

Gegenüber dem Basis-Notebook kommt ein weiteres Feld hinzu: `active_skills` 
speichert, welche Analyst-Skills für diesen Durchlauf geladen wurden. 
Das ist wichtig, damit der Checker weiß, *wogegen* er bewerten soll.

In [ ]:
class AnalysisState(TypedDict):
    """
    State-Schema mit Skills und Qualitätsbewertung.
    """
    # ── Eingabe ──
    file_path: str
    
    # ── Data Profiler → Analyst ──
    data_profile: dict
    active_skills: list[str]      # Welche Analyst-Skills geladen sind
    
    # ── Analyst ⇄ Tools ──
    messages: Annotated[list, add_messages]
    
    # ── Quality Checker ──
    quality_score: float
    quality_feedback: str
    revision_count: int

print("✓ AnalysisState definiert")

## 5. Tool: Python-Sandbox

In [ ]:
@tool
def execute_python(code: str) -> str:
    """Führt Python-Code aus und gibt stdout + stderr zurück.
    
    Verfügbare Bibliotheken: pandas (als pd), matplotlib.pyplot (als plt),
    seaborn (als sns), numpy (als np).
    
    WICHTIG: Alle Plots mit plt.savefig('plots/plot_name.png') speichern,
    NICHT plt.show() verwenden. Print-Ausgaben werden zurückgegeben.
    """
    import numpy as np
    
    namespace = {"pd": pd, "np": np, "plt": plt, "sns": sns}
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout_capture), \
             contextlib.redirect_stderr(stderr_capture):
            exec(code, namespace)

        output = stdout_capture.getvalue()
        errors = stderr_capture.getvalue()
        result = output if output else "(Kein Output – nutze print())"
        if errors:
            result += f"\n\nWarnungen:\n{errors}"
        return result

    except Exception as e:
        return f"FEHLER:\n{type(e).__name__}: {e}"

print("✓ Tool 'execute_python' definiert")

## 6. Knoten definieren

### 6a. Data Profiler (+ Skill-Auswahl)

Der Profiler hat jetzt eine zusätzliche Aufgabe: Basierend auf dem Datenprofil 
wählt er die passenden Analyst-Skills aus. Diese Entscheidung ist 
deterministisch (kein LLM nötig) und wird im State als `active_skills` gespeichert.

In [ ]:
def data_profiler(state: AnalysisState) -> dict:
    """
    Knoten 1: Datei einlesen, Profil erstellen, Skills auswählen.
    """
    file_path = state["file_path"]
    
    if file_path.endswith((".xlsx", ".xls")):
        df = pd.read_excel(file_path)
    else:
        try:
            df = pd.read_csv(file_path)
        except Exception:
            df = pd.read_csv(file_path, sep=";")
    
    profile = {
        "dateiname": os.path.basename(file_path),
        "zeilen": int(df.shape[0]),
        "spalten": int(df.shape[1]),
        "spaltennamen": list(df.columns),
        "datentypen": {col: str(dtype) for col, dtype in df.dtypes.items()},
        "fehlende_werte": {
            col: int(count) for col, count in df.isnull().sum().items() if count > 0
        },
        "numerische_spalten": list(df.select_dtypes(include="number").columns),
        "kategorische_spalten": list(df.select_dtypes(include=["object", "category"]).columns),
        "grundstatistik": json.loads(df.describe(include="all").to_json()),
        "stichprobe_5_zeilen": json.loads(df.head().to_json(orient="records", date_format="iso")),
    }
    
    # ── Skill-Auswahl ──
    skills = select_analyst_skills(profile)
    print(f"📊 Datenprofil: {profile['zeilen']} Zeilen, {profile['spalten']} Spalten")
    print(f"🧩 Aktive Skills: {', '.join(skills) if skills else 'nur Basis'}")
    
    return {
        "data_profile": profile,
        "active_skills": skills,
    }

print("✓ Knoten 'data_profiler' definiert (mit Skill-Auswahl)")

### 6b. Analyst (Skill-gesteuert)

Der Analyst baut seinen System-Prompt jetzt aus den aktiven Skills zusammen, 
statt einen monolithischen String zu verwenden.

In [ ]:
def analyst(state: AnalysisState) -> dict:
    """
    Knoten 2: LLM-Agent mit skill-basiertem System-Prompt.
    """
    profile = state["data_profile"]
    file_path = state["file_path"]
    active_skills = state.get("active_skills", [])
    revision = state.get("revision_count", 0)
    
    # ── System-Prompt aus Skills zusammenbauen ──
    system_prompt = build_analyst_prompt(profile, file_path, active_skills)
    
    llm_with_tools = llm_analyst.bind_tools([execute_python])

    # Erster Durchlauf
    if not state.get("messages"):
        initial_messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content="Bitte analysiere den Datensatz."),
        ]
        response = llm_with_tools.invoke(initial_messages)
        return {"messages": initial_messages + [response]}
    
    # Revision: Checker-Feedback injizieren
    if revision > 0 and state.get("quality_feedback"):
        feedback_msg = HumanMessage(content=(
            f"--- QUALITÄTS-REVIEW (Runde {revision}/{MAX_REVISIONS}) ---\n\n"
            f"Ein Reviewer hat deine bisherige Analyse bewertet und folgende "
            f"Verbesserungen angefordert:\n\n"
            f"{state['quality_feedback']}\n\n"
            f"Bitte führe NUR die fehlenden Analysen durch. "
            f"Wiederhole NICHT, was bereits gut war."
        ))
        response = llm_with_tools.invoke(state["messages"] + [feedback_msg])
        return {"messages": [feedback_msg, response]}
    
    # Normaler Folgeaufruf (nach Tool-Ergebnis)
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

print("✓ Knoten 'analyst' definiert (skill-gesteuert)")

### 6c. Quality Checker (bewertet gegen aktive Skills)

Der Checker weiß jetzt, welche Skills der Analyst hatte, und bewertet 
gezielt dagegen: Wenn z.B. `zeitreihe.md` aktiv war, aber der Analyst 
keine Zeitreihenanalyse gemacht hat, senkt das den Vollständigkeits-Score.

In [ ]:
def quality_checker(state: AnalysisState) -> dict:
    """
    Knoten 3: Qualitätsbewertung gegen die aktiven Skills.
    """
    profile = state["data_profile"]
    active_skills = state.get("active_skills", [])
    
    # Konversation aufbereiten
    conversation = ""
    for msg in state["messages"]:
        role = getattr(msg, "type", "unknown")
        content = getattr(msg, "content", "")
        if content and role in ("ai", "human", "tool"):
            conversation += f"[{role.upper()}]: {content[:1500]}\n\n"
    
    # ── Checker-Prompt aus Skill laden ──
    checker_prompt = build_checker_prompt(profile, active_skills, conversation)

    response = llm_checker.invoke([
        SystemMessage(content="Du antwortest ausschließlich mit validem JSON. Kein Markdown, kein Text davor oder danach."),
        HumanMessage(content=checker_prompt),
    ])
    
    # ── JSON parsen (robust) ──
    raw = response.content.strip()
    json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw, re.DOTALL)
    
    if json_match:
        try:
            result = json.loads(json_match.group())
        except json.JSONDecodeError:
            result = {"completeness": 0.5, "depth": 0.5, "missing": ["JSON-Parsing fehlgeschlagen"], "strengths": []}
    else:
        result = {"completeness": 0.5, "depth": 0.5, "missing": ["Keine valide Antwort vom Checker"], "strengths": []}
    
    # ── Gewichteten Score berechnen ──
    completeness = max(0.0, min(1.0, float(result.get("completeness", 0.5))))
    depth = max(0.0, min(1.0, float(result.get("depth", 0.5))))
    weighted_score = (WEIGHT_COMPLETENESS * completeness) + (WEIGHT_DEPTH * depth)
    
    # ── Feedback zusammenbauen ──
    missing = result.get("missing", [])
    strengths = result.get("strengths", [])
    
    feedback_parts = []
    feedback_parts.append(f"SCORE: {weighted_score:.0%} (Threshold: {QUALITY_THRESHOLD:.0%})")
    feedback_parts.append(f"  Vollständigkeit: {completeness:.0%} (Gewicht: {WEIGHT_COMPLETENESS:.0%})")
    feedback_parts.append(f"  Erkenntnistiefe: {depth:.0%} (Gewicht: {WEIGHT_DEPTH:.0%})")
    feedback_parts.append(f"  Geprüft gegen Skills: {', '.join(active_skills) if active_skills else 'nur Basis'}")
    
    if strengths:
        feedback_parts.append(f"\nSTÄRKEN: {'; '.join(strengths)}")
    if missing:
        feedback_parts.append(f"\nFEHLT: {'; '.join(missing)}")
    
    feedback = "\n".join(feedback_parts)
    
    revision = state.get("revision_count", 0)
    status = "✅ AKZEPTIERT" if weighted_score >= QUALITY_THRESHOLD else f"🔄 NACHBESSERUNG ({revision + 1}/{MAX_REVISIONS})"
    print(f"\n{'='*50}")
    print(f"QUALITY CHECK — {status}")
    print(f"{'='*50}")
    print(feedback)
    print(f"{'='*50}\n")
    
    return {
        "quality_score": weighted_score,
        "quality_feedback": feedback,
    }

print("✓ Knoten 'quality_checker' definiert (skill-aware)")

## 7. Routing & Revisionszähler

In [ ]:
def should_use_tool(state: AnalysisState) -> str:
    """Analyst → Tools oder Quality Check"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "quality_checker"


def should_revise(state: AnalysisState) -> str:
    """Quality Check → END oder Revision"""
    score = state.get("quality_score", 0.0)
    revision = state.get("revision_count", 0)
    
    if score >= QUALITY_THRESHOLD:
        print(f"✅ Analyse akzeptiert (Score: {score:.0%} ≥ {QUALITY_THRESHOLD:.0%})")
        return END
    
    if revision >= MAX_REVISIONS:
        print(f"⚠️ Max. Revisionen erreicht ({MAX_REVISIONS}×) — Score: {score:.0%}")
        return END
    
    print(f"🔄 Nachbesserung angefordert (Score: {score:.0%} < {QUALITY_THRESHOLD:.0%})")
    return "analyst"


def increment_revision(state: AnalysisState) -> dict:
    """Zählt den Revisionszähler hoch."""
    return {"revision_count": state.get("revision_count", 0) + 1}

print("✓ Routing-Funktionen definiert")

## 8. Graph zusammenbauen

```
START ──→ data_profiler ──→ analyst
                              │
                        conditional_edge
                         ╱           ╲
                       tools    quality_checker
                         │              │
                       edge       conditional_edge
                         │         ╱           ╲
                         └──→ analyst          END
                               ↑
                         increment_revision
```

In [ ]:
def build_graph():
    """Baut den LangGraph-Graphen mit Reflexionsschleife und Skills."""
    
    graph = StateGraph(AnalysisState)
    
    graph.add_node("data_profiler", data_profiler)
    graph.add_node("analyst", analyst)
    graph.add_node("tools", ToolNode([execute_python]))
    graph.add_node("quality_checker", quality_checker)
    graph.add_node("increment_revision", increment_revision)
    
    graph.add_edge(START, "data_profiler")
    graph.add_edge("data_profiler", "analyst")
    graph.add_edge("tools", "analyst")
    graph.add_edge("increment_revision", "analyst")
    
    graph.add_conditional_edges(
        "analyst", should_use_tool,
        path_map={"tools": "tools", "quality_checker": "quality_checker"}
    )
    
    graph.add_conditional_edges(
        "quality_checker", should_revise,
        path_map={END: END, "analyst": "increment_revision"}
    )
    
    return graph.compile()

app = build_graph()
print("✓ Graph kompiliert")

### Graph visualisieren

In [ ]:
import base64
from IPython import display

def display_graph(graph_app):
    try:
        png_data = graph_app.get_graph().draw_mermaid_png()
        display.display(display.Image(data=png_data))
    except Exception:
        mermaid_code = graph_app.get_graph().draw_mermaid()
        print(mermaid_code)

display_graph(app)

## 9. Ausführungsfunktion

In [ ]:
def run_analysis(file_path: str):
    """
    Generator: Yielded nach jedem Graph-Schritt den aktuellen
    Zwischenbericht, sodass Gradio live aktualisieren kann.
    """
    app = build_graph()

    initial_state = {
        "file_path": file_path,
        "data_profile": {},
        "active_skills": [],
        "messages": [],
        "quality_score": 0.0,
        "quality_feedback": "",
        "revision_count": 0,
    }

    log_lines = []
    code_blocks = []
    final_text = ""
    quality_log = []

    def _build_report(status_emoji="⏳", status_text="Analyse läuft..."):
        """Baut den Markdown-Bericht aus dem aktuellen Stand zusammen."""
        report = f"### {status_emoji} {status_text}\n\n"
        report += "## 📋 Analyseverlauf\n\n"
        report += "\n\n".join(log_lines) if log_lines else "*Starte...*"

        if quality_log:
            report += "\n\n---\n\n## 🔍 Qualitätsbewertung\n\n"
            report += "\n\n".join(quality_log)

        if final_text:
            report += "\n\n---\n\n## 📊 Ergebnisse & Erkenntnisse\n\n"
            report += final_text

        if code_blocks:
            report += "\n\n---\n\n## 🐍 Verwendeter Analyse-Code\n\n"
            for i, code in enumerate(code_blocks, 1):
                report += f"### Schritt {i}\n\n```python\n{code}\n```\n\n"

        return report

    for step in app.stream(initial_state, stream_mode="updates"):
        node_name = list(step.keys())[0]

        if node_name == "data_profiler":
            profile = step[node_name].get("data_profile", {})
            skills = step[node_name].get("active_skills", [])
            log_lines.append(
                f"📊 **Datenprofil:** "
                f"{profile.get('zeilen', '?')} Zeilen, "
                f"{profile.get('spalten', '?')} Spalten"
            )
            num = profile.get("numerische_spalten", [])
            kat = profile.get("kategorische_spalten", [])
            if num:
                log_lines.append(f"  - Numerisch: {', '.join(num)}")
            if kat:
                log_lines.append(f"  - Kategorisch: {', '.join(kat)}")
            if profile.get("fehlende_werte"):
                log_lines.append(f"  - ⚠️ Fehlende Werte: {profile['fehlende_werte']}")
            if skills:
                log_lines.append(f"  - 🧩 Skills: {', '.join(skills)}")
            yield _build_report("📊", "Datenprofil erstellt — Analyst startet...")

        elif node_name == "analyst":
            msgs = step[node_name].get("messages", [])
            for msg in (msgs if isinstance(msgs, list) else [msgs]):
                if hasattr(msg, "tool_calls") and msg.tool_calls:
                    for tc in msg.tool_calls:
                        full_code = tc["args"]["code"]
                        code_preview = full_code[:80].replace("\n", " ")
                        log_lines.append(f"🔧 Code: `{code_preview}...`")
                        code_blocks.append(full_code)
                elif hasattr(msg, "content") and msg.content:
                    if hasattr(msg, "type") and msg.type == "ai":
                        final_text = msg.content
            yield _build_report("🤖", "Analyst arbeitet...")

        elif node_name == "tools":
            for msg in step[node_name]["messages"]:
                if msg.content and not msg.content.startswith("(Kein"):
                    preview = msg.content[:120].replace("\n", " ")
                    log_lines.append(f"✅ Ergebnis: `{preview}`")
            yield _build_report("⚙️", "Code wird ausgeführt...")

        elif node_name == "quality_checker":
            score = step[node_name].get("quality_score", 0)
            feedback = step[node_name].get("quality_feedback", "")
            quality_log.append(f"🔍 **Quality Score: {score:.0%}**")
            quality_log.append(feedback)
            yield _build_report("🔍", f"Quality Check: {score:.0%}")

        elif node_name == "increment_revision":
            rev = step[node_name].get("revision_count", 0)
            log_lines.append(f"\n🔄 **Nachbesserung {rev}/{MAX_REVISIONS} gestartet**\n")
            yield _build_report("🔄", f"Nachbesserung {rev}/{MAX_REVISIONS}...")

    # ── Finaler Bericht ──
    yield _build_report("✅", "Analyse abgeschlossen")

## 10. Beispieldaten & Schnelltest

In [ ]:
beispiel_csv = "beispiel_verkaufsdaten.csv"

if not os.path.exists(beispiel_csv):
    daten = [
        "datum,produkt,kategorie,region,umsatz,menge,rabatt_prozent,kundenzufriedenheit",
        "2024-01-05,Widget Pro,Elektronik,Nord,1250.00,25,5,4.2",
        "2024-01-08,Widget Pro,Elektronik,Süd,980.50,18,10,3.8",
        "2024-01-12,Gadget Mini,Elektronik,West,2340.00,65,0,4.5",
        "2024-01-15,Office Chair,Möbel,Nord,890.00,10,15,4.0",
        "2024-01-18,Widget Pro,Elektronik,Ost,1100.00,22,5,4.1",
        "2024-01-22,Standing Desk,Möbel,Süd,3200.00,8,0,4.8",
        "2024-01-25,Gadget Mini,Elektronik,Nord,1890.50,52,8,4.3",
        "2024-02-01,Office Chair,Möbel,West,445.00,5,20,3.5",
        "2024-02-05,Widget Pro,Elektronik,Süd,2100.00,42,0,4.6",
        "2024-02-08,Kabelset,Zubehör,Ost,320.00,80,0,3.9",
        "2024-02-12,Standing Desk,Möbel,Nord,4800.00,12,5,4.7",
        "2024-02-15,Gadget Mini,Elektronik,West,1560.00,43,12,3.6",
        "2024-02-18,Widget Pro,Elektronik,Ost,870.00,16,15,3.4",
        "2024-02-22,Kabelset,Zubehör,Süd,480.00,120,0,4.0",
        "2024-02-25,Office Chair,Möbel,Nord,1780.00,20,8,4.4",
        "2024-03-01,Standing Desk,Möbel,West,1600.00,4,0,4.9",
        "2024-03-05,Gadget Mini,Elektronik,Nord,3120.00,87,5,4.2",
        "2024-03-08,Widget Pro,Elektronik,Süd,1650.00,33,10,3.7",
        "2024-03-12,Kabelset,Zubehör,Ost,560.00,140,3,4.1",
        "2024-03-15,Office Chair,Möbel,West,,15,10,",
        "2024-03-18,Standing Desk,Möbel,Süd,2400.00,6,0,4.6",
        "2024-03-22,Widget Pro,Elektronik,Nord,1920.00,38,5,4.3",
        "2024-03-25,Gadget Mini,Elektronik,Ost,1240.00,34,8,3.9",
        "2024-03-28,Kabelset,Zubehör,Nord,280.00,70,0,4.0",
        "2024-04-02,Standing Desk,Möbel,West,4000.00,10,0,4.8",
        "2024-04-05,Widget Pro,Elektronik,Süd,,28,12,3.5",
        "2024-04-08,Office Chair,Möbel,Ost,1335.00,15,5,4.2",
        "2024-04-12,Gadget Mini,Elektronik,Nord,2730.00,76,3,4.4",
        "2024-04-15,Kabelset,Zubehör,Süd,640.00,160,0,3.8",
        "2024-04-18,Standing Desk,Möbel,Nord,3600.00,9,8,4.5",
    ]
    with open(beispiel_csv, "w") as f:
        f.write("\n".join(daten))
    print(f"✓ Beispieldatei '{beispiel_csv}' erzeugt")
else:
    print(f"✓ Beispieldatei '{beispiel_csv}' bereits vorhanden")

In [ ]:
# ── Schnelltest (ohne Gradio) ──
# result = run_analysis(beispiel_csv)
# print(result)

## 11. Gradio-Interface

In [ ]:
import gradio as gr


def gradio_analyze(file):
    if file is None:
        yield "⚠️ Bitte lade eine CSV- oder Excel-Datei hoch."
        return

    file_path = file.name if hasattr(file, "name") else str(file)

    try:
        yield "### ⏳ Analyse wird gestartet...\n\n*Bitte warten — der Agent arbeitet.*"
        for update in run_analysis(file_path):
            yield update
    except Exception as e:
        yield f"❌ **Fehler:**\n\n`{type(e).__name__}: {e}`"


demo = gr.Interface(
    fn=gradio_analyze,
    inputs=gr.File(
        label="📁 CSV- oder Excel-Datei hochladen",
        file_types=[".csv", ".xlsx", ".xls", ".tsv"],
    ),
    outputs=gr.Markdown(label="📊 Analyseergebnis"),
    title="🔬 Datenanalyse-Agent (Stufe 2 – Skills + Reflexion)",
    description=(
        "Lade eine CSV- oder Excel-Datei hoch. Der Agent wählt automatisch "
        "passende Analyse-Skills, führt die Analyse durch und ein Qualitäts-Agent "
        f"bewertet die Ergebnisse (Threshold: {QUALITY_THRESHOLD:.0%}, "
        f"max. {MAX_REVISIONS} Revisionen)."
    ),
    examples=[[beispiel_csv]] if os.path.exists(beispiel_csv) else None,
    flagging_mode="never",
)

print("✓ Gradio-Interface definiert (mit Live-Streaming)")

### Interface starten

In [ ]:
demo.launch(
    # share=True,
    # server_name="0.0.0.0",
    # server_port=7860,
)

---

## 12. Eigene Skills erstellen

### Schritt 1: Markdown-Datei anlegen

Erstelle eine neue `.md`-Datei in `skills/analyst/`:

```markdown
<!-- skills/analyst/ecommerce.md -->
# E-Commerce-Analyse

Der Datensatz enthält E-Commerce-Daten.

## Zusätzliche Analyseaufgaben

- **Umsatz pro Kunde**: Durchschnittlicher Warenkorbwert
- **Produktperformance**: Top/Flop-Produkte nach Umsatz und Menge
- **Rabatt-Analyse**: Korrelation zwischen Rabatt und Kundenzufriedenheit
```

### Schritt 2: Skill-Auswahl anpassen

In `select_analyst_skills()` eine Bedingung ergänzen:

```python
# E-Commerce erkennen (Spaltenname enthält 'kunde', 'warenkorb', 'bestellung')
ecommerce_keywords = {"kunde", "warenkorb", "bestellung", "order", "cart"}
if any(keyword in col for col in all_cols for keyword in ecommerce_keywords):
    skills.append("ecommerce.md")
```

### Schritt 3: Checker-Skill erweitern (optional)

Falls der Checker den neuen Skill auch kennen soll, die Beschreibung 
in `skills/checker/standard.md` ergänzen.

---

## 13. Ausblick: Stufe 3 → Human-in-the-Loop

**Stufe 3:** Checkpointing + Human-Approval vor dem End-Report

**Stufe 4:** Report-Generator + Produktionshärtung